# Task

> Objective: GOAL: Quantify cross-cancer alteration frequencies of miRNA biogenesis pathway genes using TCGA data from cBioPortal to support a Figure 3b-style overview and identify gaps requiring COSMIC supplementation. DATASETS: Download public cBioPortal TCGA PanCancer Atlas studies and extract alteration profiles for key genes including DROSHA, DGCR8, DICER1, TARBP2, PRKRA, XPO5, AGO1-4, LIN28A, LIN28B, SMAD2, SMAD4, and TP53-linked pathway context where relevant. OUTPUT: A gene-by-cancer matrix/table of alteration frequencies with alteration class breakdowns, ranked highlights, and a short note on tumour types or genes with limited TCGA coverage.

> Datasets:

## Rationale

The request requires programmatic retrieval and processing of TCGA PanCancer Atlas data from cBioPortal, computing per-gene per-cancer alteration frequencies with class-level breakdowns and generating a matrix/report; these tasks need code to download, aggregate, and compute accurate frequency tables across many studies.

## Step 0: Explore working directory and identify any existing documentation/data files

Before downloading TCGA/cBioPortal data, inspect the local environment for any provided datasets, prior downloads, or documentation that define schema/expectations.

In [1]:

import os

# Set working directory
os.chdir('/home/user')
print(f"Current working directory: {os.getcwd()}\n")

# List top-level files/directories
print("=" * 60)
print("TOP-LEVEL FILES/DIRECTORIES (ls -lah)")
print("=" * 60)
!ls -lah

print("\n" + "=" * 60)
print("DIRECTORY SIZES (du -sh *)")
print("=" * 60)
!du -sh * 2>/dev/null | head -30

print("\n" + "=" * 60)
print("DISK FREE (df -h)")
print("=" * 60)
!df -h

print("\n" + "=" * 60)
print("MEMORY (free -m)")
print("=" * 60)
!free -m

# Search for documentation-like files up to depth 3
print("\n" + "=" * 60)
print("DOCUMENTATION FILES (depth ≤ 3)")
print("=" * 60)

import subprocess

extensions = ['*.md', '*.txt', '*.pdf', '*.doc', '*.docx', '*.rtf']
find_args = []
for i, ext in enumerate(extensions):
    if i > 0:
        find_args.append('-o')
    find_args.extend(['-name', ext])

cmd = ['find', '.', '-maxdepth', '3', '-type', 'f', '('] + find_args + [')']
result = subprocess.run(cmd, capture_output=True, text=True, encoding='latin-1')

files_found = [f.strip() for f in result.stdout.strip().split('\n') if f.strip()]

if files_found:
    print(f"\n{'Path':<70} {'Size':>10}")
    print("-" * 82)
    for fpath in sorted(files_found):
        try:
            size = os.path.getsize(fpath)
            if size >= 1024 * 1024:
                size_str = f"{size / (1024*1024):.1f} MB"
            elif size >= 1024:
                size_str = f"{size / 1024:.1f} KB"
            else:
                size_str = f"{size} B"
            print(f"{fpath:<70} {size_str:>10}")
        except OSError:
            print(f"{fpath:<70} {'N/A':>10}")
    print(f"\nTotal documentation files found: {len(files_found)}")
else:
    print("No documentation files found.")


Current working directory: /home/user

TOP-LEVEL FILES/DIRECTORIES (ls -lah)
total 24K
drwx------ 2 user user 4.0K Jan 26 19:21 .
drwxr-xr-x 3 root root 4.0K Jan 26 19:19 ..
-rw-r--r-- 1 user user  220 Jul 30  2025 .bash_logout
-rw-r--r-- 1 user user 3.5K Jul 30  2025 .bashrc
-rw-r--r-- 1 user user  807 Jul 30  2025 .profile
-rw-r--r-- 1 user user    0 Jan 26 19:20 .sudo_as_admin_successful
-rw-r--r-- 1 user user  176 Jan 26 19:21 .wget-hsts



DIRECTORY SIZES (du -sh *)



DISK FREE (df -h)
Filesystem      Size  Used Avail Use% Mounted on
/dev/root        54G   16G   38G  30% /
devtmpfs        3.9G     0  3.9G   0% /dev
tmpfs           3.9G     0  3.9G   0% /dev/shm
tmpfs           1.6G   80K  1.6G   1% /run
tmpfs           5.0M     0  5.0M   0% /run/lock
tmpfs           3.9G     0  3.9G   0% /tmp



MEMORY (free -m)
               total        used        free      shared  buff/cache   available
Mem:            7964         786        6829           0         629        7178
Swap:              0           0           0



DOCUMENTATION FILES (depth ≤ 3)
No documentation files found.


## Step 1: Identify all TCGA PanCancer Atlas studies available via the cBioPortal public API

Use the cBioPortal REST API to enumerate studies, then filter to TCGA PanCancer Atlas cohorts (one per tumour type). Capture study IDs, names, and cancer type identifiers so we know exactly which datasets to query for gene alteration frequencies in later steps.

In [2]:

import requests
import pandas as pd

BASE_URL = "https://www.cbioportal.org/api"
HEADERS = {"Accept": "application/json"}

# 1) Test connectivity
print("Testing connectivity...")
resp = requests.get(f"{BASE_URL}/info", headers=HEADERS, timeout=30)
print(f"  /api/info status: {resp.status_code}")
if resp.status_code == 200:
    info = resp.json()
    print(f"  Portal version: {info.get('portalVersion', 'N/A')}")
    print(f"  DB version: {info.get('dbVersion', 'N/A')}")

# 2) Download full studies list via pagination
print("\nDownloading studies list...")
all_studies = []
page_number = 0
page_size = 1000

while True:
    params = {
        "projection": "SUMMARY",
        "pageSize": page_size,
        "pageNumber": page_number,
        "direction": "ASC",
        "sortBy": "studyId"
    }
    resp = requests.get(f"{BASE_URL}/studies", headers=HEADERS, params=params, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        break
    all_studies.extend(data)
    print(f"  Page {page_number}: {len(data)} studies retrieved")
    page_number += 1
    if len(data) < page_size:
        break

print(f"\nTotal studies retrieved: {len(all_studies)}")

# 3) Create full dataframe and filter for TCGA PanCancer Atlas
df_all = pd.DataFrame(all_studies)
print(f"Columns available: {list(df_all.columns)}")

# Filter: name contains 'PanCancer Atlas' AND 'TCGA' (case-insensitive)
# OR studyId matches pattern *_tcga_pan_can_atlas_2018
mask_name = (
    df_all['name'].str.contains('PanCancer Atlas', case=False, na=False) &
    df_all['name'].str.contains('TCGA', case=False, na=False)
)
mask_id = df_all['studyId'].str.contains('_tcga_pan_can_atlas_2018', case=False, na=False)

mask = mask_name | mask_id

# 4) Create filtered dataframe
cols = ['studyId', 'name', 'cancerTypeId']
if 'description' in df_all.columns:
    cols.append('description')

df_pancan = df_all.loc[mask, cols].sort_values('studyId').reset_index(drop=True)

print(f"\nNumber of TCGA PanCancer Atlas studies: {len(df_pancan)}")
print("\n" + "=" * 100)
print("TCGA PanCancer Atlas Studies")
print("=" * 100)

# Display full table
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', 200)

# Print without description for readability, then show description separately
print(df_pancan[['studyId', 'name', 'cancerTypeId']].to_string(index=True))

# 5) Save to CSV
output_path = "/home/user/tcga_pancan_atlas_studies.csv"
df_pancan.to_csv(output_path, index=False)
print(f"\nSaved filtered studies to: {output_path}")
print(f"File size: {os.path.getsize(output_path)} bytes")


Testing connectivity...


  /api/info status: 200
  Portal version: v6.4.4
  DB version: 2.14.5



  Page 0: 535 studies retrieved

Total studies retrieved: 535


Columns available: ['name', 'description', 'publicStudy', 'pmid', 'citation', 'groups', 'status', 'importDate', 'allSampleCount', 'readPermission', 'resourceCounts', 'studyId', 'cancerTypeId', 'referenceGenome']



Number of TCGA PanCancer Atlas studies: 32

TCGA PanCancer Atlas Studies
                             studyId                                                           name cancerTypeId
0        acc_tcga_pan_can_atlas_2018               Adrenocortical Carcinoma (TCGA, PanCancer Atlas)          acc
1       blca_tcga_pan_can_atlas_2018           Bladder Urothelial Carcinoma (TCGA, PanCancer Atlas)         blca
2       brca_tcga_pan_can_atlas_2018              Breast Invasive Carcinoma (TCGA, PanCancer Atlas)         brca
3       cesc_tcga_pan_can_atlas_2018       Cervical Squamous Cell Carcinoma (TCGA, PanCancer Atlas)         cesc
4       chol_tcga_pan_can_atlas_2018                     Cholangiocarcinoma (TCGA, PanCancer Atlas)         chol
5   coadread_tcga_pan_can_atlas_2018              Colorectal Adenocarcinoma (TCGA, PanCancer Atlas)     coadread
6       dlbc_tcga_pan_can_atlas_2018          Diffuse Large B-Cell Lymphoma (TCGA, PanCancer Atlas)     dlbclnos
7       esca_tcga_pan_

## Step 2: Characterize cBioPortal data modalities for one representative TCGA PanCancer Atlas study (BRCA)

Query the BRCA PanCancer Atlas study for available sample lists and molecular profiles to identify the correct IDs for (i) the full cohort denominator and (ii) mutation and discrete CNA (GISTIC) alteration data; this establishes the exact API objects needed before scaling extraction across all 32 studies.

In [4]:

import json

STUDY_ID = "brca_tcga_pan_can_atlas_2018"

# ============================================================
# 1) Sample Lists
# ============================================================
print("=" * 80)
print(f"1) Sample Lists for {STUDY_ID}")
print("=" * 80)

resp_sl = requests.get(f"{BASE_URL}/studies/{STUDY_ID}/sample-lists", headers=HEADERS, timeout=60)
resp_sl.raise_for_status()
df_sample_lists = pd.DataFrame(resp_sl.json())

print(f"\nNumber of sample lists: {len(df_sample_lists)}")
print(f"Columns: {list(df_sample_lists.columns)}")

# sampleCount is not a direct field — we need to get it from the sampleIds
# Fetch sampleIds for each sample list to compute sampleCount
sample_counts = []
for sl_id in df_sample_lists['sampleListId']:
    resp_ids = requests.get(
        f"{BASE_URL}/sample-lists/{sl_id}/sample-ids",
        headers=HEADERS, timeout=60
    )
    resp_ids.raise_for_status()
    sample_counts.append(len(resp_ids.json()))

df_sample_lists['sampleCount'] = sample_counts

# Display table
display_cols = ['sampleListId', 'name', 'category', 'sampleCount']
display_cols = [c for c in display_cols if c in df_sample_lists.columns]
print(f"\n{df_sample_lists[display_cols].to_string(index=True)}")

# ============================================================
# 2) Choose 'all samples' sampleListId
# ============================================================
print("\n" + "=" * 80)
print("2) Choosing 'all samples' sampleListId")
print("=" * 80)

# Strategy: prefer category containing 'all_cases_in_study', then name contains 'All samples'
mask_all_cat = df_sample_lists['category'].str.contains('all_cases_in_study', case=False, na=False)

if mask_all_cat.any():
    candidates = df_sample_lists[mask_all_cat]
    idx_max = candidates['sampleCount'].idxmax()
    chosen_sl = candidates.loc[idx_max, 'sampleListId']
    chosen_count = int(candidates.loc[idx_max, 'sampleCount'])
else:
    # Fallback: name contains 'all' with max sample count
    mask_all_name = df_sample_lists['name'].str.contains('all', case=False, na=False)
    if mask_all_name.any():
        candidates = df_sample_lists[mask_all_name]
    else:
        candidates = df_sample_lists
    idx_max = candidates['sampleCount'].idxmax()
    chosen_sl = candidates.loc[idx_max, 'sampleListId']
    chosen_count = int(candidates.loc[idx_max, 'sampleCount'])

print(f"\n  Chosen sampleListId: {chosen_sl}")
print(f"  Sample count:        {chosen_count}")

# ============================================================
# 3) Molecular Profiles
# ============================================================
print("\n" + "=" * 80)
print(f"3) Molecular Profiles for {STUDY_ID}")
print("=" * 80)

resp_mp = requests.get(f"{BASE_URL}/studies/{STUDY_ID}/molecular-profiles", headers=HEADERS, timeout=60)
resp_mp.raise_for_status()
df_mol_profiles = pd.DataFrame(resp_mp.json())

print(f"\nNumber of molecular profiles: {len(df_mol_profiles)}")
print(f"Columns: {list(df_mol_profiles.columns)}")

mp_display_cols = ['molecularProfileId', 'name', 'molecularAlterationType', 'datatype']
mp_display_cols = [c for c in mp_display_cols if c in df_mol_profiles.columns]
print(f"\n{df_mol_profiles[mp_display_cols].to_string(index=True)}")

# ============================================================
# 4) Identify candidate mutation and CNA profile IDs
# ============================================================
print("\n" + "=" * 80)
print("4) Candidate Mutation & CNA Profile IDs")
print("=" * 80)

# Mutation candidates
mask_mut = (
    df_mol_profiles['molecularAlterationType'].str.contains('MUTATION', case=False, na=False) |
    df_mol_profiles['molecularProfileId'].str.contains('mutation', case=False, na=False) |
    df_mol_profiles['name'].str.contains('mutation', case=False, na=False)
)
mutation_ids = df_mol_profiles.loc[mask_mut, 'molecularProfileId'].tolist()
print(f"\n  Mutation profile IDs: {mutation_ids}")

# CNA candidates (discrete / GISTIC)
mask_cna = (
    df_mol_profiles['molecularAlterationType'].str.contains('COPY_NUMBER', case=False, na=False) &
    (
        df_mol_profiles['molecularProfileId'].str.contains('gistic|cna', case=False, na=False) |
        df_mol_profiles['name'].str.contains('gistic|cna|copy|putative', case=False, na=False) |
        df_mol_profiles['datatype'].str.contains('DISCRETE', case=False, na=False)
    )
)
cna_ids = df_mol_profiles.loc[mask_cna, 'molecularProfileId'].tolist()
print(f"  CNA profile IDs:      {cna_ids}")

# If no discrete CNA found, fall back to all COPY_NUMBER profiles
if not cna_ids:
    mask_cna_all = df_mol_profiles['molecularAlterationType'].str.contains('COPY_NUMBER', case=False, na=False)
    cna_ids = df_mol_profiles.loc[mask_cna_all, 'molecularProfileId'].tolist()
    print(f"  (Fallback) All CNA profile IDs: {cna_ids}")

# ============================================================
# 5) Save JSON
# ============================================================
print("\n" + "=" * 80)
print("5) Saving profile IDs to JSON")
print("=" * 80)

output_json = {
    "studyId": STUDY_ID,
    "chosen_all_sampleListId": chosen_sl,
    "chosen_all_sampleCount": chosen_count,
    "candidate_mutation_profile_ids": mutation_ids,
    "candidate_cna_profile_ids": cna_ids
}

json_path = "/home/user/brca_pan_can_atlas_profile_ids.json"
with open(json_path, 'w') as f:
    json.dump(output_json, f, indent=2)

print(f"\nSaved to: {json_path}")
print(f"File size: {os.path.getsize(json_path)} bytes")
print(f"\nContents:")
print(json.dumps(output_json, indent=2))


1) Sample Lists for brca_tcga_pan_can_atlas_2018



Number of sample lists: 11
Columns: ['category', 'name', 'description', 'sampleListId', 'studyId']



                                                  sampleListId                                                                          name                                       category  sampleCount
0                   brca_tcga_pan_can_atlas_2018_3way_complete                                                              Complete samples  all_cases_with_mutation_and_cna_and_mrna_data          994
1                             brca_tcga_pan_can_atlas_2018_all                                                                   All samples                             all_cases_in_study         1084
2                             brca_tcga_pan_can_atlas_2018_cna                                                         Samples with CNA data                        all_cases_with_cna_data         1070
3                          brca_tcga_pan_can_atlas_2018_cnaseq                                            Samples with mutation and CNA data           all_cases_with_mutation_and_cna_data        


Number of molecular profiles: 16
Columns: ['molecularAlterationType', 'genericAssayType', 'datatype', 'name', 'description', 'showProfileInAnalysisTab', 'patientLevel', 'molecularProfileId', 'studyId', 'sortOrder', 'pivotThreshold']

                                                                   molecularProfileId                                                                       name molecularAlterationType     datatype
0                                           brca_tcga_pan_can_atlas_2018_armlevel_cna                                 Putative arm-level copy-number from GISTIC           GENERIC_ASSAY  CATEGORICAL
1                                       brca_tcga_pan_can_atlas_2018_genetic_ancestry                                                           Genetic Ancestry           GENERIC_ASSAY  LIMIT-VALUE
2                                                 brca_tcga_pan_can_atlas_2018_gistic                               Putative copy-number alterations from GISTIC  COPY_NUMB

## Step 3: Enumerate mutation/CNA molecular profiles and appropriate sample lists for all TCGA PanCancer Atlas studies

To compute cross-cancer alteration frequencies consistently, we first need (per study) the mutation (MAF) and discrete CNA (GISTIC) profile IDs and the best-matching sampleListIds/denominators (e.g., sequenced for mutations, cna for CNAs, plus all). We will also flag studies missing any required modality as potential TCGA coverage gaps.

In [5]:

# Enumerate mutation/CNA molecular profiles and sample lists for all TCGA PanCancer Atlas studies

import time

# Load the studies list
df_studies = pd.read_csv("/home/user/tcga_pancan_atlas_studies.csv")
print(f"Loaded {len(df_studies)} TCGA PanCancer Atlas studies\n")

results = []

for idx, row in df_studies.iterrows():
    sid = row['studyId']
    cancer_type = row['cancerTypeId']
    study_name = row['name']
    print(f"[{idx+1}/{len(df_studies)}] {sid} ... ", end="", flush=True)
    
    # --- Molecular Profiles ---
    try:
        resp_mp = requests.get(f"{BASE_URL}/studies/{sid}/molecular-profiles", headers=HEADERS, timeout=60)
        resp_mp.raise_for_status()
        mol_profiles = pd.DataFrame(resp_mp.json())
    except Exception as e:
        print(f"ERROR (mol profiles): {e}")
        mol_profiles = pd.DataFrame()
    
    # Mutation profile: molecularAlterationType == 'MUTATION_EXTENDED', datatype == 'MAF'
    mut_id = None
    if len(mol_profiles) > 0:
        mask_mut = (
            (mol_profiles['molecularAlterationType'] == 'MUTATION_EXTENDED') &
            (mol_profiles['datatype'] == 'MAF')
        )
        if mask_mut.any():
            mut_id = mol_profiles.loc[mask_mut, 'molecularProfileId'].iloc[0]
    
    # Discrete CNA (GISTIC): molecularAlterationType == 'COPY_NUMBER_ALTERATION', datatype == 'DISCRETE'
    gistic_id = None
    if len(mol_profiles) > 0:
        mask_cna = (
            (mol_profiles['molecularAlterationType'] == 'COPY_NUMBER_ALTERATION') &
            (mol_profiles['datatype'] == 'DISCRETE')
        )
        if mask_cna.any():
            gistic_id = mol_profiles.loc[mask_cna, 'molecularProfileId'].iloc[0]
    
    # Structural variant profile
    sv_id = None
    if len(mol_profiles) > 0:
        mask_sv = (mol_profiles['molecularAlterationType'] == 'STRUCTURAL_VARIANT')
        if mask_sv.any():
            sv_id = mol_profiles.loc[mask_sv, 'molecularProfileId'].iloc[0]
    
    # --- Sample Lists ---
    try:
        resp_sl = requests.get(f"{BASE_URL}/studies/{sid}/sample-lists", headers=HEADERS, timeout=60)
        resp_sl.raise_for_status()
        sample_lists = pd.DataFrame(resp_sl.json())
    except Exception as e:
        print(f"ERROR (sample lists): {e}")
        sample_lists = pd.DataFrame()
    
    # For each sample list, get sample count
    sl_counts = {}
    if len(sample_lists) > 0:
        for _, sl_row in sample_lists.iterrows():
            sl_id = sl_row['sampleListId']
            try:
                resp_ids = requests.get(f"{BASE_URL}/sample-lists/{sl_id}/sample-ids", headers=HEADERS, timeout=60)
                resp_ids.raise_for_status()
                sl_counts[sl_id] = len(resp_ids.json())
            except:
                sl_counts[sl_id] = 0
    
    # Choose best sample list IDs
    # 'all' - contains '_all'
    sl_all_id = None
    sl_all_count = 0
    if len(sample_lists) > 0:
        mask_all = sample_lists['sampleListId'].str.contains('_all$', case=False, na=False)
        if mask_all.any():
            candidates = sample_lists.loc[mask_all, 'sampleListId'].tolist()
            # Pick the one with max count
            best = max(candidates, key=lambda x: sl_counts.get(x, 0))
            sl_all_id = best
            sl_all_count = sl_counts.get(best, 0)
    
    # 'sequenced' - contains '_sequenced'
    sl_seq_id = None
    sl_seq_count = 0
    if len(sample_lists) > 0:
        mask_seq = sample_lists['sampleListId'].str.contains('_sequenced', case=False, na=False)
        if mask_seq.any():
            candidates = sample_lists.loc[mask_seq, 'sampleListId'].tolist()
            best = max(candidates, key=lambda x: sl_counts.get(x, 0))
            sl_seq_id = best
            sl_seq_count = sl_counts.get(best, 0)
    
    # 'cna' - contains '_cna'
    sl_cna_id = None
    sl_cna_count = 0
    if len(sample_lists) > 0:
        mask_cna_sl = sample_lists['sampleListId'].str.contains('_cna', case=False, na=False)
        if mask_cna_sl.any():
            candidates = sample_lists.loc[mask_cna_sl, 'sampleListId'].tolist()
            best = max(candidates, key=lambda x: sl_counts.get(x, 0))
            sl_cna_id = best
            sl_cna_count = sl_counts.get(best, 0)
    
    rec = {
        'studyId': sid,
        'cancerTypeId': cancer_type,
        'studyName': study_name,
        'sampleListId_all': sl_all_id,
        'sampleCount_all': sl_all_count,
        'sampleListId_sequenced': sl_seq_id,
        'sampleCount_sequenced': sl_seq_count,
        'sampleListId_cna': sl_cna_id,
        'sampleCount_cna': sl_cna_count,
        'mutationProfileId': mut_id,
        'gisticProfileId': gistic_id,
        'svProfileId': sv_id,
        'has_mutation': mut_id is not None,
        'has_gistic': gistic_id is not None,
        'has_sv': sv_id is not None,
        'missing_mutation': mut_id is None,
        'missing_gistic': gistic_id is None,
    }
    results.append(rec)
    
    status_parts = []
    if mut_id: status_parts.append("MUT")
    if gistic_id: status_parts.append("CNA")
    if sv_id: status_parts.append("SV")
    print(f"OK [{'/'.join(status_parts)}] all={sl_all_count} seq={sl_seq_count} cna={sl_cna_count}")
    
    # Small delay to be nice to the API
    time.sleep(0.1)

# Build summary DataFrame
df_profile_map = pd.DataFrame(results)

# Print compact summary
print("\n" + "=" * 100)
print("SUMMARY")
print("=" * 100)
n_total = len(df_profile_map)
n_mut = df_profile_map['has_mutation'].sum()
n_gistic = df_profile_map['has_gistic'].sum()
n_sv = df_profile_map['has_sv'].sum()
n_both = ((df_profile_map['has_mutation']) & (df_profile_map['has_gistic'])).sum()

print(f"Total studies:                   {n_total}")
print(f"Studies with mutation profile:   {n_mut}")
print(f"Studies with GISTIC profile:     {n_gistic}")
print(f"Studies with SV profile:         {n_sv}")
print(f"Studies with BOTH mut & gistic:  {n_both}")

# List studies missing mutation
missing_mut = df_profile_map[df_profile_map['missing_mutation']]
if len(missing_mut) > 0:
    print(f"\nStudies MISSING mutation profile ({len(missing_mut)}):")
    for _, r in missing_mut.iterrows():
        print(f"  - {r['studyId']} ({r['studyName']})")
else:
    print("\nAll studies have a mutation profile.")

# List studies missing gistic
missing_gistic = df_profile_map[df_profile_map['missing_gistic']]
if len(missing_gistic) > 0:
    print(f"\nStudies MISSING GISTIC profile ({len(missing_gistic)}):")
    for _, r in missing_gistic.iterrows():
        print(f"  - {r['studyId']} ({r['studyName']})")
else:
    print("\nAll studies have a GISTIC profile.")

# Display the table compactly
print("\n" + "=" * 100)
print("PROFILE MAP TABLE")
print("=" * 100)
display_cols = ['studyId', 'cancerTypeId', 'sampleCount_all', 'sampleCount_sequenced', 
                'sampleCount_cna', 'mutationProfileId', 'gisticProfileId', 'has_sv']
print(df_profile_map[display_cols].to_string(index=False))

# Save to CSV
out_path = "/home/user/tcga_pancan_atlas_profile_map.csv"
df_profile_map.to_csv(out_path, index=False)
print(f"\nSaved full mapping table to: {out_path}")
print(f"File size: {os.path.getsize(out_path)} bytes")


Loaded 32 TCGA PanCancer Atlas studies

[1/32] acc_tcga_pan_can_atlas_2018 ... 

OK [MUT/CNA/SV] all=92 seq=91 cna=89
[2/32] blca_tcga_pan_can_atlas_2018 ... 

OK [MUT/CNA/SV] all=411 seq=410 cna=408
[3/32] brca_tcga_pan_can_atlas_2018 ... 

OK [MUT/CNA/SV] all=1084 seq=1066 cna=1070
[4/32] cesc_tcga_pan_can_atlas_2018 ... 

OK [MUT/CNA/SV] all=297 seq=291 cna=293
[5/32] chol_tcga_pan_can_atlas_2018 ... 

...

OK [MUT/CNA/SV] all=500 seq=490 cna=497
[29/32] thym_tcga_pan_can_atlas_2018 ... 

OK [MUT/CNA/SV] all=123 seq=123 cna=123
[30/32] ucec_tcga_pan_can_atlas_2018 ... 

OK [MUT/CNA/SV] all=529 seq=517 cna=523
[31/32] ucs_tcga_pan_can_atlas_2018 ... 

OK [MUT/CNA/SV] all=57 seq=57 cna=56
[32/32] uvm_tcga_pan_can_atlas_2018 ... 

OK [MUT/CNA/SV] all=80 seq=80 cna=80

SUMMARY
Total studies:                   32
Studies with mutation profile:   32
Studies with GISTIC profile:     32
Studies with SV profile:         32
Studies with BOTH mut & gistic:  32

All studies have a mutation profile.

All studies have a GISTIC profile.

PROFILE MAP TABLE
                         studyId cancerTypeId  sampleCount_all  sampleCount_sequenced  sampleCount_cna                          mutationProfileId                         gisticProfileId  has_sv
     acc_tcga_pan_can_atlas_2018          acc               92                     91               89      acc_tcga_pan_can_atlas_2018_mutations      acc_tcga_pan_can_atlas_2018_gistic    True
    blca_tcga_pan_can_atlas_2018         blca              411                    410              408     blca_tcga_pan_can_atlas_2018_mutations     blca_tcga_pan_can_atlas_2018_gistic    True
    brca_tcga_pan_can_atlas_2018         brca             1084                   1066             1

## Step 4: Prototype alteration-frequency extraction for miRNA biogenesis pathway genes in one TCGA PanCancer Atlas study (BRCA)

Before scaling across all 32 cancers, validate the cBioPortal API workflow end-to-end: map the target gene symbols to Entrez IDs, choose an appropriate BRCA sample set with both mutation and CNA data (prefer a CNASEQ/overlap set), fetch mutation (MUTATION_EXTENDED) and GISTIC discrete CNA data, then compute per-gene alteration frequencies with class breakdowns (mutation, amplification/gain, shallow/deep deletion) and an overall ‘any alteration’ frequency.

In [7]:

# Step: Prototype alteration-frequency extraction for miRNA biogenesis pathway genes in BRCA

import time

# ============================================================
# 1) Load profile map and extract BRCA row
# ============================================================
df_pm = pd.read_csv("/home/user/tcga_pancan_atlas_profile_map.csv")
print(f"Profile map columns: {list(df_pm.columns)}\n")

brca_row = df_pm[df_pm['studyId'] == 'brca_tcga_pan_can_atlas_2018'].iloc[0]
print("BRCA row:")
print(brca_row.to_string())

mutation_profile_id = brca_row['mutationProfileId']
gistic_profile_id = brca_row['gisticProfileId']
print(f"\nMutation profile ID: {mutation_profile_id}")
print(f"GISTIC profile ID:  {gistic_profile_id}")

# ============================================================
# 2) Sample Lists for BRCA
# ============================================================
print("\n" + "=" * 80)
print("2) Sample Lists for BRCA")
print("=" * 80)

STUDY_ID = "brca_tcga_pan_can_atlas_2018"

resp_sl = requests.get(f"{BASE_URL}/studies/{STUDY_ID}/sample-lists", headers=HEADERS, timeout=60)
resp_sl.raise_for_status()
sample_lists = pd.DataFrame(resp_sl.json())

# Identify key sample lists
sl_map = {}
for suffix in ['_all', '_sequenced', '_cna', '_cnaseq']:
    mask = sample_lists['sampleListId'].str.endswith(suffix)
    if mask.any():
        sl_map[suffix] = sample_lists.loc[mask, 'sampleListId'].iloc[0]

print("\nIdentified sample lists:")
for k, v in sl_map.items():
    print(f"  {k}: {v}")

# Fetch sample IDs for each
sl_samples = {}
for suffix, sl_id in sl_map.items():
    resp = requests.get(f"{BASE_URL}/sample-lists/{sl_id}/sample-ids", headers=HEADERS, timeout=60)
    resp.raise_for_status()
    sl_samples[suffix] = set(resp.json())
    print(f"  {suffix} count: {len(sl_samples[suffix])}")

# Set relationships
if '_sequenced' in sl_samples and '_cna' in sl_samples:
    seq_set = sl_samples['_sequenced']
    cna_set = sl_samples['_cna']
    intersection = seq_set & cna_set
    print(f"\n  |sequenced ∩ cna| = {len(intersection)}")

if '_cnaseq' in sl_samples:
    cnaseq_set = sl_samples['_cnaseq']
    is_sub_seq = cnaseq_set <= seq_set
    is_sub_cna = cnaseq_set <= cna_set
    print(f"  cnaseq ⊆ sequenced? {is_sub_seq}")
    print(f"  cnaseq ⊆ cna?       {is_sub_cna}")

# Choose analysis set
if '_cnaseq' in sl_samples and is_sub_seq and is_sub_cna:
    analysis_sample_list_id = sl_map['_cnaseq']
    analysis_N = len(sl_samples['_cnaseq'])
    analysis_sample_ids = sl_samples['_cnaseq']
    use_sample_list = True
    print(f"\n→ Using cnaseq sample list: {analysis_sample_list_id} (N={analysis_N})")
else:
    analysis_sample_ids = intersection
    analysis_N = len(intersection)
    use_sample_list = False
    print(f"\n→ Using sequenced ∩ cna intersection (N={analysis_N})")

# ============================================================
# 3) Target genes
# ============================================================
TARGET_GENES = ['DROSHA','DGCR8','DICER1','TARBP2','PRKRA','XPO5',
                'AGO1','AGO2','AGO3','AGO4','LIN28A','LIN28B',
                'SMAD2','SMAD4','TP53']
print(f"\n3) Target genes ({len(TARGET_GENES)}): {TARGET_GENES}")

# ============================================================
# 4) Gene Symbol → Entrez ID Mapping
# ============================================================
print("\n" + "=" * 80)
print("4) Gene Symbol → Entrez ID Mapping")
print("=" * 80)

gene_map = {}
not_found = []
for sym in TARGET_GENES:
    resp = requests.get(f"{BASE_URL}/genes/{sym}", headers=HEADERS, timeout=30)
    if resp.status_code == 200:
        gdata = resp.json()
        gene_map[sym] = gdata['entrezGeneId']
    else:
        # Fallback: keyword search
        resp2 = requests.get(f"{BASE_URL}/genes", headers=HEADERS,
                             params={"keyword": sym, "projection": "SUMMARY", "pageSize": 100}, timeout=30)
        if resp2.status_code == 200:
            matches = [g for g in resp2.json() if g.get('hugoGeneSymbol') == sym]
            if matches:
                gene_map[sym] = matches[0]['entrezGeneId']
            else:
                not_found.append(sym)
        else:
            not_found.append(sym)
    time.sleep(0.05)

print(f"\n{'Symbol':<15} {'EntrezGeneId':>12}")
print("-" * 28)
for sym in TARGET_GENES:
    if sym in gene_map:
        print(f"{sym:<15} {gene_map[sym]:>12}")

if not_found:
    print(f"\nNOT FOUND: {not_found}")
else:
    print(f"\nAll {len(TARGET_GENES)} symbols mapped successfully.")

entrez_ids = [gene_map[s] for s in TARGET_GENES if s in gene_map]

# ============================================================
# 5) Fetch Mutation Data
# ============================================================
print("\n" + "=" * 80)
print("5) Fetching Mutation Data")
print("=" * 80)

mut_url = f"{BASE_URL}/molecular-profiles/{mutation_profile_id}/mutations/fetch"
mut_body = {"entrezGeneIds": entrez_ids}
if use_sample_list:
    mut_body["sampleListId"] = analysis_sample_list_id
else:
    mut_body["sampleIds"] = sorted(list(analysis_sample_ids))

resp_mut = requests.post(mut_url, headers={**HEADERS, "Content-Type": "application/json"},
                         json=mut_body, params={"projection": "SUMMARY"}, timeout=120)
resp_mut.raise_for_status()
mut_data = resp_mut.json()
df_mut = pd.DataFrame(mut_data)
print(f"Mutation records fetched: {len(df_mut)}")
if len(df_mut) > 0:
    print(f"Columns: {list(df_mut.columns)}")

# Detect gene symbol column
if len(df_mut) > 0:
    # Find the correct column for gene symbol
    gene_col_candidates = [c for c in df_mut.columns if 'hugo' in c.lower() or 'genesymbol' in c.lower() or 'gene' in c.lower()]
    print(f"Gene-related columns: {gene_col_candidates}")
    
    # Check if nested 'gene' dict exists
    if 'gene' in df_mut.columns and isinstance(df_mut['gene'].iloc[0], dict):
        df_mut['hugoGeneSymbol'] = df_mut['gene'].apply(lambda x: x.get('hugoGeneSymbol', ''))
        gene_col_mut = 'hugoGeneSymbol'
    elif 'hugoGeneSymbol' in df_mut.columns:
        gene_col_mut = 'hugoGeneSymbol'
    else:
        # Use entrezGeneId and reverse map
        gene_col_mut = None
    
    sample_col_mut = 'sampleId' if 'sampleId' in df_mut.columns else 'uniqueSampleKey'
    
    if gene_col_mut:
        print(f"Using gene column: {gene_col_mut}, sample column: {sample_col_mut}")
    else:
        print("WARNING: No gene symbol column found; will use entrezGeneId")
    
    # Build reverse map: entrezGeneId -> symbol
    entrez_to_sym = {v: k for k, v in gene_map.items()}
    
    # Per-gene: set of unique sampleIds with ≥1 mutation
    if gene_col_mut:
        mut_per_gene = df_mut.groupby(gene_col_mut)[sample_col_mut].apply(set).to_dict()
    else:
        df_mut['_sym'] = df_mut['entrezGeneId'].map(entrez_to_sym)
        mut_per_gene = df_mut.groupby('_sym')[sample_col_mut].apply(set).to_dict()
    
    print("\nMutated sample counts per gene:")
    for g in TARGET_GENES:
        n = len(mut_per_gene.get(g, set()))
        print(f"  {g:<12} {n:>4} samples")
else:
    mut_per_gene = {}
    print("No mutation records found.")

# ============================================================
# 6) Fetch Discrete CNA (GISTIC) Data
# ============================================================
print("\n" + "=" * 80)
print("6) Fetching GISTIC CNA Data")
print("=" * 80)

cna_url = f"{BASE_URL}/molecular-profiles/{gistic_profile_id}/molecular-data/fetch"
cna_body = {"entrezGeneIds": entrez_ids}
if use_sample_list:
    cna_body["sampleListId"] = analysis_sample_list_id
else:
    cna_body["sampleIds"] = sorted(list(analysis_sample_ids))

resp_cna = requests.post(cna_url, headers={**HEADERS, "Content-Type": "application/json"},
                         json=cna_body, params={"projection": "SUMMARY"}, timeout=120)
resp_cna.raise_for_status()
cna_data = resp_cna.json()
df_cna = pd.DataFrame(cna_data)
print(f"CNA records fetched: {len(df_cna)}")
if len(df_cna) > 0:
    print(f"Columns: {list(df_cna.columns)}")
    
    # Detect gene symbol column for CNA
    if 'gene' in df_cna.columns and isinstance(df_cna['gene'].iloc[0], dict):
        df_cna['hugoGeneSymbol'] = df_cna['gene'].apply(lambda x: x.get('hugoGeneSymbol', ''))
        gene_col_cna = 'hugoGeneSymbol'
    elif 'hugoGeneSymbol' in df_cna.columns:
        gene_col_cna = 'hugoGeneSymbol'
    else:
        df_cna['_sym'] = df_cna['entrezGeneId'].map(entrez_to_sym)
        gene_col_cna = '_sym'
    
    sample_col_cna = 'sampleId' if 'sampleId' in df_cna.columns else 'uniqueSampleKey'
    
    # Ensure value is numeric
    df_cna['value'] = pd.to_numeric(df_cna['value'], errors='coerce').astype('Int64')
    
    print(f"\nCNA value distribution:")
    print(df_cna['value'].value_counts().sort_index())
    
    # Per gene CNA stats
    cna_stats = {}
    for g in TARGET_GENES:
        gdf = df_cna[df_cna[gene_col_cna] == g]
        cna_stats[g] = {
            'amp_n': int((gdf['value'] == 2).sum()),
            'gain_n': int((gdf['value'] == 1).sum()),
            'shallow_del_n': int((gdf['value'] == -1).sum()),
            'deep_del_n': int((gdf['value'] == -2).sum()),
            'any_cna_n': int((gdf['value'] != 0).sum()),
            'any_cna_samples': set(gdf.loc[gdf['value'] != 0, sample_col_cna])
        }
    
    print("\nCNA counts per gene:")
    print(f"  {'Gene':<12} {'Amp':>5} {'Gain':>5} {'ShDel':>5} {'DpDel':>5} {'AnyCNA':>6}")
    for g in TARGET_GENES:
        s = cna_stats[g]
        print(f"  {g:<12} {s['amp_n']:>5} {s['gain_n']:>5} {s['shallow_del_n']:>5} {s['deep_del_n']:>5} {s['any_cna_n']:>6}")
else:
    cna_stats = {}
    print("No CNA records found.")

# ============================================================
# 7) Build per-gene summary table
# ============================================================
print("\n" + "=" * 80)
print("7) Per-Gene Summary Table")
print("=" * 80)

N = analysis_N
print(f"Denominator (N): {N} samples (cnaseq list)")

rows = []
for g in TARGET_GENES:
    mut_samples = mut_per_gene.get(g, set())
    cna_samples = cna_stats.get(g, {}).get('any_cna_samples', set())
    # Filter to analysis samples only
    mut_samples_in = mut_samples & analysis_sample_ids
    cna_samples_in = cna_samples & analysis_sample_ids
    any_alter_samples = mut_samples_in | cna_samples_in
    
    cs = cna_stats.get(g, {})
    rows.append({
        'gene': g,
        'entrez_id': gene_map.get(g, ''),
        'mut_n': len(mut_samples_in),
        'amp_n': cs.get('amp_n', 0),
        'gain_n': cs.get('gain_n', 0),
        'shallow_del_n': cs.get('shallow_del_n', 0),
        'deep_del_n': cs.get('deep_del_n', 0),
        'any_cna_n': cs.get('any_cna_n', 0),
        'any_alter_n': len(any_alter_samples),
        'N': N,
        'mut_freq': len(mut_samples_in) / N,
        'amp_freq': cs.get('amp_n', 0) / N,
        'deep_del_freq': cs.get('deep_del_n', 0) / N,
        'any_cna_freq': cs.get('any_cna_n', 0) / N,
        'any_alter_freq': len(any_alter_samples) / N,
    })

df_summary = pd.DataFrame(rows).sort_values('any_alter_freq', ascending=False).reset_index(drop=True)

print("\nTop 10 genes by any_alter_freq:")
print(df_summary.head(10).to_string(index=False))

# Save
out_path = "/home/user/brca_miRNA_biogenesis_alteration_frequencies.csv"
df_summary.to_csv(out_path, index=False)
print(f"\nSaved full table ({len(df_summary)} genes) to: {out_path}")
print(f"File size: {os.path.getsize(out_path)} bytes")


Profile map columns: ['studyId', 'cancerTypeId', 'studyName', 'sampleListId_all', 'sampleCount_all', 'sampleListId_sequenced', 'sampleCount_sequenced', 'sampleListId_cna', 'sampleCount_cna', 'mutationProfileId', 'gisticProfileId', 'svProfileId', 'has_mutation', 'has_gistic', 'has_sv', 'missing_mutation', 'missing_gistic']

BRCA row:
studyId                                        brca_tcga_pan_can_atlas_2018
cancerTypeId                                                           brca
studyName                 Breast Invasive Carcinoma (TCGA, PanCancer Atlas)
sampleListId_all                           brca_tcga_pan_can_atlas_2018_all
sampleCount_all                                                        1084
sampleListId_sequenced               brca_tcga_pan_can_atlas_2018_sequenced
sampleCount_sequenced                                                  1066
sampleListId_cna                           brca_tcga_pan_can_atlas_2018_cna
sampleCount_cna                                          


Identified sample lists:
  _all: brca_tcga_pan_can_atlas_2018_all
  _sequenced: brca_tcga_pan_can_atlas_2018_sequenced
  _cna: brca_tcga_pan_can_atlas_2018_cna
  _cnaseq: brca_tcga_pan_can_atlas_2018_cnaseq


  _all count: 1084


  _sequenced count: 1066


  _cna count: 1070


  _cnaseq count: 996

  |sequenced ∩ cna| = 1052
  cnaseq ⊆ sequenced? True
  cnaseq ⊆ cna?       True

→ Using cnaseq sample list: brca_tcga_pan_can_atlas_2018_cnaseq (N=996)

3) Target genes (15): ['DROSHA', 'DGCR8', 'DICER1', 'TARBP2', 'PRKRA', 'XPO5', 'AGO1', 'AGO2', 'AGO3', 'AGO4', 'LIN28A', 'LIN28B', 'SMAD2', 'SMAD4', 'TP53']

4) Gene Symbol → Entrez ID Mapping



Symbol          EntrezGeneId
----------------------------
DROSHA                 29102
DGCR8                  54487
DICER1                 23405
TARBP2                  6895
PRKRA                   8575
XPO5                   57510
AGO1                   26523
AGO2                   27161
AGO3                  192669
AGO4                  192670
LIN28A                 79727
LIN28B                389421
SMAD2                   4087
SMAD4                   4089
TP53                    7157

All 15 symbols mapped successfully.

5) Fetching Mutation Data


Mutation records fetched: 454
Columns: ['uniqueSampleKey', 'uniquePatientKey', 'molecularProfileId', 'sampleId', 'patientId', 'entrezGeneId', 'studyId', 'center', 'mutationStatus', 'validationStatus', 'tumorAltCount', 'tumorRefCount', 'normalAltCount', 'normalRefCount', 'startPosition', 'endPosition', 'referenceAllele', 'proteinChange', 'mutationType', 'ncbiBuild', 'variantType', 'keyword', 'chr', 'variantAllele', 'refseqMrnaId', 'proteinPosStart', 'proteinPosEnd']
Gene-related columns: ['entrezGeneId']

Mutated sample counts per gene:
  DROSHA          9 samples
  DGCR8           5 samples
  DICER1         11 samples
  TARBP2          7 samples
  PRKRA           3 samples
  XPO5           12 samples
  AGO1           10 samples
  AGO2            8 samples
  AGO3            8 samples
  AGO4            8 samples
  LIN28A          2 samples
  LIN28B          2 samples
  SMAD2           6 samples
  SMAD4           9 samples
  TP53          344 samples

6) Fetching GISTIC CNA Data


CNA records fetched: 14940
Columns: ['uniqueSampleKey', 'uniquePatientKey', 'entrezGeneId', 'molecularProfileId', 'sampleId', 'patientId', 'studyId', 'value']

CNA value distribution:
value
-2      33
-1    4282
0     7989
1     2427
2      209
Name: count, dtype: Int64

CNA counts per gene:
  Gene           Amp  Gain ShDel DpDel AnyCNA
  DROSHA          14   318    95     0    427
  DGCR8            7   120   425     2    554
  DICER1           7   130   300     3    440
  TARBP2           3   190   155     0    348
  PRKRA            3    91   210     0    304
  XPO5            13   227   139     3    382
  AGO1             6   117   308     0    431
  AGO2           113   470    60     3    646
  AGO3             9   116   305     0    430
  AGO4             7   114   311     0    432
  LIN28A           1    58   413     1    473
  LIN28B          20   150   313     2    485
  SMAD2            4   142   319     3    468
  SMAD4            2   137   327     7    473
  TP53           

## Step 5: Define a consistent denominator (samples with both mutation and CNA data) across all 32 TCGA PanCancer Atlas studies

To avoid cross-cancer bias from missing modality data, compute per-study sample sets representing the intersection of the study’s “sequenced” and “cna” sample lists (i.e., cases with both mutation and GISTIC CNA coverage). Also record whether pre-defined combined lists (e.g., cnaseq, 3way_complete) exist and how restrictive they are, then save the intersection sample IDs for downstream alteration-frequency extraction.

In [8]:

import time
import json
import numpy as np

# Load profile map
df_pm = pd.read_csv("/home/user/tcga_pancan_atlas_profile_map.csv")
print(f"Loaded profile map: {len(df_pm)} studies")
print(f"Columns: {list(df_pm.columns)}\n")

# Create a session for connection pooling
session = requests.Session()
session.headers.update({"Accept": "application/json"})

def fetch_sample_ids(sample_list_id):
    """Fetch sample IDs for a given sample list ID. Returns a set."""
    if pd.isna(sample_list_id) or sample_list_id == '' or sample_list_id is None:
        return set()
    resp = session.get(f"{BASE_URL}/sample-lists/{sample_list_id}", timeout=60)
    resp.raise_for_status()
    data = resp.json()
    sample_ids = data.get('sampleIds', [])
    # Handle list vs newline-delimited string
    if isinstance(sample_ids, str):
        sample_ids = [s.strip() for s in sample_ids.split('\n') if s.strip()]
    elif isinstance(sample_ids, list):
        # Could be list of strings or something else
        sample_ids = [str(s).strip() for s in sample_ids if str(s).strip()]
    return set(sample_ids)

# Storage for results
results = []
intersection_map = {}  # studyId -> list of intersection sample IDs

for idx, row in df_pm.iterrows():
    sid = row['studyId']
    cancer_type = row['cancerTypeId']
    study_name = row.get('studyName', row.get('name', sid))
    
    sl_seq = row.get('sampleListId_sequenced', None)
    sl_cna = row.get('sampleListId_cna', None)
    sc_all = row.get('sampleCount_all', 0)
    sc_seq = row.get('sampleCount_sequenced', 0)
    sc_cna = row.get('sampleCount_cna', 0)
    
    print(f"[{idx+1}/{len(df_pm)}] {sid} ... ", end="", flush=True)
    
    # 1) Fetch sequenced and CNA sample IDs, compute intersection
    try:
        seq_samples = fetch_sample_ids(sl_seq)
    except Exception as e:
        print(f"ERR seq: {e}", end=" ")
        seq_samples = set()
    
    try:
        cna_samples = fetch_sample_ids(sl_cna)
    except Exception as e:
        print(f"ERR cna: {e}", end=" ")
        cna_samples = set()
    
    intersection = seq_samples & cna_samples
    intersection_n = len(intersection)
    intersection_map[sid] = sorted(list(intersection))
    
    # 2) List all sample lists for this study, find cnaseq and 3way_complete
    cnaseq_sl_id = None
    cnaseq_n = None
    threeway_sl_id = None
    threeway_n = None
    
    try:
        resp_sls = session.get(f"{BASE_URL}/sample-lists", params={"studyId": sid}, timeout=60)
        resp_sls.raise_for_status()
        all_sls = resp_sls.json()
        
        for sl in all_sls:
            sl_id = sl.get('sampleListId', '')
            sl_id_lower = sl_id.lower()
            if 'cnaseq' in sl_id_lower and cnaseq_sl_id is None:
                cnaseq_sl_id = sl_id
            if '3way_complete' in sl_id_lower and threeway_sl_id is None:
                threeway_sl_id = sl_id
        
        # Fetch sample IDs for cnaseq
        if cnaseq_sl_id:
            try:
                cnaseq_samples = fetch_sample_ids(cnaseq_sl_id)
                cnaseq_n = len(cnaseq_samples)
            except Exception as e:
                print(f"ERR cnaseq: {e}", end=" ")
        
        # Fetch sample IDs for 3way_complete
        if threeway_sl_id:
            try:
                threeway_samples = fetch_sample_ids(threeway_sl_id)
                threeway_n = len(threeway_samples)
            except Exception as e:
                print(f"ERR 3way: {e}", end=" ")
    except Exception as e:
        print(f"ERR sample lists: {e}", end=" ")
    
    # Compute ratios
    min_seq_cna = min(len(seq_samples), len(cna_samples)) if (len(seq_samples) > 0 and len(cna_samples) > 0) else 0
    ratio_intersection_min = intersection_n / min_seq_cna if min_seq_cna > 0 else np.nan
    ratio_cnaseq_intersection = cnaseq_n / intersection_n if (cnaseq_n is not None and intersection_n > 0) else np.nan
    ratio_threeway_intersection = threeway_n / intersection_n if (threeway_n is not None and intersection_n > 0) else np.nan
    
    rec = {
        'studyId': sid,
        'cancerTypeId': cancer_type,
        'studyName': study_name,
        'sampleCount_all': sc_all,
        'sampleCount_sequenced': len(seq_samples),
        'sampleCount_cna': len(cna_samples),
        'intersection_sequenced_cna_n': intersection_n,
        'cnaseq_sampleListId': cnaseq_sl_id if cnaseq_sl_id else np.nan,
        'cnaseq_n': cnaseq_n if cnaseq_n is not None else np.nan,
        'threeway_sampleListId': threeway_sl_id if threeway_sl_id else np.nan,
        'threeway_n': threeway_n if threeway_n is not None else np.nan,
        'ratio_intersection_over_min_seq_cna': ratio_intersection_min,
        'ratio_cnaseq_over_intersection': ratio_cnaseq_intersection,
        'ratio_threeway_over_intersection': ratio_threeway_intersection,
    }
    results.append(rec)
    
    parts = [f"seq={len(seq_samples)}", f"cna={len(cna_samples)}", f"∩={intersection_n}"]
    if cnaseq_n is not None:
        parts.append(f"cnaseq={cnaseq_n}")
    if threeway_n is not None:
        parts.append(f"3way={threeway_n}")
    print(f"OK [{', '.join(parts)}]")
    
    time.sleep(0.1)

# Build summary DataFrame
df_sample_summary = pd.DataFrame(results)

# 4) Save CSV and JSON
csv_path = "/home/user/tcga_pancan_atlas_sample_set_summary.csv"
df_sample_summary.to_csv(csv_path, index=False)
print(f"\nSaved summary CSV to: {csv_path}")
print(f"File size: {os.path.getsize(csv_path)} bytes")

json_path = "/home/user/tcga_pancan_atlas_sequenced_cna_intersection_sampleIds.json"
with open(json_path, 'w') as f:
    json.dump(intersection_map, f)
print(f"Saved intersection sample IDs JSON to: {json_path}")
print(f"File size: {os.path.getsize(json_path)} bytes")

# 5) QC Report
print("\n" + "=" * 80)
print("QC REPORT")
print("=" * 80)

int_vals = df_sample_summary['intersection_sequenced_cna_n']
print(f"\n(a) intersection_sequenced_cna_n statistics:")
print(f"    Min:    {int_vals.min()}")
print(f"    Median: {int_vals.median():.0f}")
print(f"    Max:    {int_vals.max()}")
print(f"    Total:  {int_vals.sum()}")

# (b) Studies where intersection < 0.9 * min(sequenced, cna)
print(f"\n(b) Studies where intersection < 0.9 * min(sampleCount_sequenced, sampleCount_cna):")
flagged_b = []
for _, r in df_sample_summary.iterrows():
    min_sc = min(r['sampleCount_sequenced'], r['sampleCount_cna'])
    threshold = 0.9 * min_sc
    if r['intersection_sequenced_cna_n'] < threshold:
        flagged_b.append(r)
        print(f"    {r['studyId']}: intersection={r['intersection_sequenced_cna_n']}, "
              f"seq={r['sampleCount_sequenced']}, cna={r['sampleCount_cna']}, "
              f"min={min_sc}, threshold={threshold:.0f}, "
              f"ratio={r['ratio_intersection_over_min_seq_cna']:.3f}")
if not flagged_b:
    print("    None — all studies have intersection ≥ 0.9 * min(seq, cna)")

# (c) Studies where cnaseq or 3way exists but < 0.9 * intersection
print(f"\n(c) Studies where cnaseq or 3way_complete exists but is < 0.9 * intersection:")
flagged_c = []
for _, r in df_sample_summary.iterrows():
    int_n = r['intersection_sequenced_cna_n']
    threshold = 0.9 * int_n
    if not pd.isna(r['cnaseq_n']) and r['cnaseq_n'] < threshold:
        flagged_c.append(('cnaseq', r))
        print(f"    {r['studyId']}: cnaseq={int(r['cnaseq_n'])}, intersection={int_n}, "
              f"ratio={r['ratio_cnaseq_over_intersection']:.3f}")
    if not pd.isna(r['threeway_n']) and r['threeway_n'] < threshold:
        flagged_c.append(('3way', r))
        print(f"    {r['studyId']}: 3way={int(r['threeway_n'])}, intersection={int_n}, "
              f"ratio={r['ratio_threeway_over_intersection']:.3f}")
if not flagged_c:
    print("    None — all existing cnaseq/3way lists are ≥ 0.9 * intersection")

# Print compact table
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)
display_cols = ['studyId', 'cancerTypeId', 'sampleCount_all', 'sampleCount_sequenced', 
                'sampleCount_cna', 'intersection_sequenced_cna_n', 'cnaseq_n', 'threeway_n',
                'ratio_intersection_over_min_seq_cna']
print(df_sample_summary[display_cols].to_string(index=False))


Loaded profile map: 32 studies
Columns: ['studyId', 'cancerTypeId', 'studyName', 'sampleListId_all', 'sampleCount_all', 'sampleListId_sequenced', 'sampleCount_sequenced', 'sampleListId_cna', 'sampleCount_cna', 'mutationProfileId', 'gisticProfileId', 'svProfileId', 'has_mutation', 'has_gistic', 'has_sv', 'missing_mutation', 'missing_gistic']

[1/32] acc_tcga_pan_can_atlas_2018 ... 

OK [seq=91, cna=89, ∩=89, cnaseq=12, 3way=75]
[2/32] blca_tcga_pan_can_atlas_2018 ... 

OK [seq=410, cna=408, ∩=407, cnaseq=12, 3way=75]
[3/32] brca_tcga_pan_can_atlas_2018 ... 

OK [seq=1066, cna=1070, ∩=1052, cnaseq=12, 3way=75]
[4/32] cesc_tcga_pan_can_atlas_2018 ... 

OK [seq=291, cna=293, ∩=287, cnaseq=12, 3way=75]
[5/32] chol_tcga_pan_can_atlas_2018 ... 

...

OK [seq=490, cna=497, ∩=487, cnaseq=12, 3way=75]
[29/32] thym_tcga_pan_can_atlas_2018 ... 

OK [seq=123, cna=123, ∩=123, cnaseq=12, 3way=75]
[30/32] ucec_tcga_pan_can_atlas_2018 ... 

OK [seq=517, cna=523, ∩=511, cnaseq=12, 3way=75]
[31/32] ucs_tcga_pan_can_atlas_2018 ... 

OK [seq=57, cna=56, ∩=56, cnaseq=12, 3way=75]
[32/32] uvm_tcga_pan_can_atlas_2018 ... 

OK [seq=80, cna=80, ∩=80, cnaseq=12, 3way=75]

Saved summary CSV to: /home/user/tcga_pancan_atlas_sample_set_summary.csv
File size: 6730 bytes
Saved intersection sample IDs JSON to: /home/user/tcga_pancan_atlas_sequenced_cna_intersection_sampleIds.json
File size: 194790 bytes

QC REPORT

(a) intersection_sequenced_cna_n statistics:
    Min:    36
    Median: 324
    Max:    1052
    Total:  10195

(b) Studies where intersection < 0.9 * min(sampleCount_sequenced, sampleCount_cna):
    None — all studies have intersection ≥ 0.9 * min(seq, cna)

(c) Studies where cnaseq or 3way_complete exists but is < 0.9 * intersection:
    acc_tcga_pan_can_atlas_2018: cnaseq=12, intersection=89, ratio=0.135
    acc_tcga_pan_can_atlas_2018: 3way=75, intersection=89, ratio=0.843
    blca_tcga_pan_can_atlas_2018: cnaseq=12, intersection=407, ratio=0.029
    blca_tcga_pan_can_atlas_2018: 3way=75, intersection=407, ratio=0.184
    brca_tcga_pan_can_atlas_2018: cnaseq=12, intersection=1052, ratio=0.011
    b

## Step 6: Compute cross-cancer alteration frequencies for miRNA biogenesis pathway genes across all 32 TCGA PanCancer Atlas studies

Using cBioPortal mutation (MAF) and GISTIC discrete CNA profiles, aggregate per-study/per-gene alteration counts and frequencies. Apply the rule that the denominator per study is the size of the intersection of the sequenced and CNA sample sets (sequenced ∩ cna), using the precomputed intersection sampleId lists to ensure consistent mutation+CNA coverage.

In [9]:

import time
import json
import numpy as np
import pandas as pd
import requests

# Load inputs
df_pm = pd.read_csv("/home/user/tcga_pancan_atlas_profile_map.csv")
with open("/home/user/tcga_pancan_atlas_sequenced_cna_intersection_sampleIds.json", 'r') as f:
    intersection_map = json.load(f)

print(f"Profile map: {len(df_pm)} studies")
print(f"Intersection map: {len(intersection_map)} studies")

# Target genes with Entrez IDs
GENE_MAP = {
    'DROSHA': 29102, 'DGCR8': 54487, 'DICER1': 23405, 'TARBP2': 6895,
    'PRKRA': 8575, 'XPO5': 57510, 'AGO1': 26523, 'AGO2': 27161,
    'AGO3': 192669, 'AGO4': 192670, 'LIN28A': 79727, 'LIN28B': 389421,
    'SMAD2': 4087, 'SMAD4': 4089, 'TP53': 7157
}
TARGET_GENES = list(GENE_MAP.keys())
ENTREZ_IDS = list(GENE_MAP.values())
ENTREZ_TO_SYM = {v: k for k, v in GENE_MAP.items()}

BASE_URL = "https://www.cbioportal.org/api"
session = requests.Session()
session.headers.update({"Accept": "application/json", "Content-Type": "application/json"})

def api_post_with_retry(url, json_body, params=None, max_retries=3, timeout=120):
    """POST with retry/backoff."""
    for attempt in range(max_retries):
        try:
            resp = session.post(url, json=json_body, params=params, timeout=timeout)
            if resp.status_code == 429:
                wait = 2 ** (attempt + 1)
                print(f"429 rate limit, waiting {wait}s...", end=" ", flush=True)
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait = 2 ** (attempt + 1)
                print(f"ERR({e.__class__.__name__}), retry in {wait}s...", end=" ", flush=True)
                time.sleep(wait)
            else:
                raise
    return []

# Silent mutation types to exclude
SILENT_TYPES = {'silent', 'Silent'}

all_rows = []
studies_processed = 0
studies_skipped = 0

for idx, row in df_pm.iterrows():
    sid = row['studyId']
    cancer_type = row['cancerTypeId']
    study_name = row.get('studyName', row.get('name', sid))
    mut_profile = row.get('mutationProfileId', None)
    gistic_profile = row.get('gisticProfileId', None)
    
    # Get intersection sample IDs
    sample_ids = intersection_map.get(sid, [])
    N = len(sample_ids)
    
    if N == 0:
        print(f"[{idx+1}/{len(df_pm)}] {sid}: SKIP (N=0)")
        studies_skipped += 1
        continue
    
    print(f"[{idx+1}/{len(df_pm)}] {sid} (N={N}) ... ", end="", flush=True)
    
    # ---- Fetch Mutations ----
    mut_per_gene = {g: set() for g in TARGET_GENES}
    if pd.notna(mut_profile) and mut_profile:
        try:
            mut_url = f"{BASE_URL}/molecular-profiles/{mut_profile}/mutations/fetch"
            mut_body = {
                "entrezGeneIds": ENTREZ_IDS,
                "sampleIds": sample_ids
            }
            mut_data = api_post_with_retry(mut_url, mut_body, params={"projection": "DETAILED"})
            
            if mut_data:
                df_mut = pd.DataFrame(mut_data)
                
                # Extract gene symbol
                if 'gene' in df_mut.columns and len(df_mut) > 0 and isinstance(df_mut['gene'].iloc[0], dict):
                    df_mut['_sym'] = df_mut['gene'].apply(lambda x: x.get('hugoGeneSymbol', ''))
                elif 'hugoGeneSymbol' in df_mut.columns:
                    df_mut['_sym'] = df_mut['hugoGeneSymbol']
                else:
                    df_mut['_sym'] = df_mut['entrezGeneId'].map(ENTREZ_TO_SYM)
                
                # Filter: exclude germline if mutationStatus present
                if 'mutationStatus' in df_mut.columns:
                    df_mut = df_mut[~df_mut['mutationStatus'].str.lower().isin(['germline'])]
                
                # Filter: exclude Silent mutations
                mut_type_col = None
                for c in ['mutationType', 'variantClassification']:
                    if c in df_mut.columns:
                        mut_type_col = c
                        break
                if mut_type_col is None:
                    # Check common column names
                    for c in df_mut.columns:
                        if 'mutation_type' in c.lower() or 'variant_classification' in c.lower():
                            mut_type_col = c
                            break
                
                if mut_type_col and len(df_mut) > 0:
                    df_mut = df_mut[~df_mut[mut_type_col].str.lower().isin(['silent'])]
                
                # Count unique samples per gene
                sample_col = 'sampleId' if 'sampleId' in df_mut.columns else 'uniqueSampleKey'
                if len(df_mut) > 0:
                    for g, grp in df_mut.groupby('_sym'):
                        if g in mut_per_gene:
                            mut_per_gene[g] = set(grp[sample_col])
                
                print(f"MUT={len(mut_data)}", end=" ", flush=True)
            else:
                print("MUT=0", end=" ", flush=True)
        except Exception as e:
            print(f"MUT_ERR({e.__class__.__name__})", end=" ", flush=True)
    else:
        print("NO_MUT_PROFILE", end=" ", flush=True)
    
    # ---- Fetch CNA (GISTIC discrete) ----
    cna_per_gene = {g: {'amp': set(), 'gain': set(), 'shdel': set(), 'dpdel': set(), 'any': set()} for g in TARGET_GENES}
    if pd.notna(gistic_profile) and gistic_profile:
        try:
            # Try discrete-copy-number-alterations endpoint first
            cna_url = f"{BASE_URL}/molecular-profiles/{gistic_profile}/discrete-copy-number-alterations/fetch"
            cna_body = {
                "entrezGeneIds": ENTREZ_IDS,
                "sampleIds": sample_ids
            }
            try:
                cna_data = api_post_with_retry(cna_url, cna_body, params={"projection": "SUMMARY", "discreteCopyNumberEventType": "ALL"})
            except:
                # Fallback to molecular-data endpoint
                cna_url = f"{BASE_URL}/molecular-profiles/{gistic_profile}/molecular-data/fetch"
                cna_data = api_post_with_retry(cna_url, cna_body, params={"projection": "SUMMARY"})
            
            if cna_data:
                df_cna = pd.DataFrame(cna_data)
                
                # Extract gene symbol
                if 'gene' in df_cna.columns and len(df_cna) > 0 and isinstance(df_cna['gene'].iloc[0], dict):
                    df_cna['_sym'] = df_cna['gene'].apply(lambda x: x.get('hugoGeneSymbol', ''))
                elif 'hugoGeneSymbol' in df_cna.columns:
                    df_cna['_sym'] = df_cna['hugoGeneSymbol']
                else:
                    df_cna['_sym'] = df_cna['entrezGeneId'].map(ENTREZ_TO_SYM)
                
                sample_col_cna = 'sampleId' if 'sampleId' in df_cna.columns else 'uniqueSampleKey'
                
                # Value column: 'value' or 'alteration'
                val_col = 'value' if 'value' in df_cna.columns else ('alteration' if 'alteration' in df_cna.columns else None)
                if val_col and len(df_cna) > 0:
                    df_cna['_val'] = pd.to_numeric(df_cna[val_col], errors='coerce').astype('Int64')
                    
                    for g, grp in df_cna.groupby('_sym'):
                        if g in cna_per_gene:
                            cna_per_gene[g]['amp'] = set(grp.loc[grp['_val'] == 2, sample_col_cna])
                            cna_per_gene[g]['gain'] = set(grp.loc[grp['_val'] == 1, sample_col_cna])
                            cna_per_gene[g]['shdel'] = set(grp.loc[grp['_val'] == -1, sample_col_cna])
                            cna_per_gene[g]['dpdel'] = set(grp.loc[grp['_val'] == -2, sample_col_cna])
                            cna_per_gene[g]['any'] = set(grp.loc[grp['_val'] != 0, sample_col_cna])
                
                print(f"CNA={len(cna_data)}", end=" ", flush=True)
            else:
                print("CNA=0", end=" ", flush=True)
        except Exception as e:
            print(f"CNA_ERR({e.__class__.__name__})", end=" ", flush=True)
    else:
        print("NO_CNA_PROFILE", end=" ", flush=True)
    
    # ---- Build per-gene rows ----
    sample_id_set = set(sample_ids)
    for g in TARGET_GENES:
        mut_samples = mut_per_gene[g] & sample_id_set
        cna_any_samples = cna_per_gene[g]['any'] & sample_id_set
        any_alter_samples = mut_samples | cna_any_samples
        mut_only = mut_samples - cna_any_samples
        cna_only = cna_any_samples - mut_samples
        mut_and_cna = mut_samples & cna_any_samples
        
        amp_n = len(cna_per_gene[g]['amp'] & sample_id_set)
        gain_n = len(cna_per_gene[g]['gain'] & sample_id_set)
        shdel_n = len(cna_per_gene[g]['shdel'] & sample_id_set)
        dpdel_n = len(cna_per_gene[g]['dpdel'] & sample_id_set)
        any_cna_n = len(cna_any_samples)
        mut_n = len(mut_samples)
        any_alter_n = len(any_alter_samples)
        
        rec = {
            'studyId': sid,
            'cancerTypeId': cancer_type,
            'studyName': study_name,
            'geneSymbol': g,
            'entrezGeneId': GENE_MAP[g],
            'N': N,
            'mut_n': mut_n,
            'amp_n': amp_n,
            'gain_n': gain_n,
            'shallow_del_n': shdel_n,
            'deep_del_n': dpdel_n,
            'any_cna_n': any_cna_n,
            'any_alter_n': any_alter_n,
            'mut_only_n': len(mut_only),
            'cna_only_n': len(cna_only),
            'mut_and_cna_n': len(mut_and_cna),
            'mut_freq': mut_n / N,
            'amp_freq': amp_n / N,
            'gain_freq': gain_n / N,
            'shallow_del_freq': shdel_n / N,
            'deep_del_freq': dpdel_n / N,
            'any_cna_freq': any_cna_n / N,
            'any_alter_freq': any_alter_n / N,
        }
        all_rows.append(rec)
    
    studies_processed += 1
    print(f"OK")
    time.sleep(0.15)

# Build long-format DataFrame
df_long = pd.DataFrame(all_rows)
print(f"\n{'='*80}")
print(f"PROCESSING COMPLETE")
print(f"{'='*80}")
print(f"Studies processed: {studies_processed}")
print(f"Studies skipped (N=0): {studies_skipped}")
print(f"Total rows in long table: {len(df_long)}")

# Save long-format CSV
long_path = "/home/user/tcga_pancan_miRNA_biogenesis_alteration_freqs_long.csv"
df_long.to_csv(long_path, index=False)
print(f"\nSaved long-format table to: {long_path}")
print(f"File size: {os.path.getsize(long_path)} bytes")

# Build wide matrix: any_alter_freq pivoted by geneSymbol x cancerTypeId
df_wide = df_long.pivot_table(
    index='geneSymbol',
    columns='cancerTypeId',
    values='any_alter_freq',
    aggfunc='first'
)
# Reorder rows by TARGET_GENES order
df_wide = df_wide.reindex([g for g in TARGET_GENES if g in df_wide.index])

wide_path = "/home/user/tcga_pancan_miRNA_biogenesis_any_alter_freq_matrix.csv"
df_wide.to_csv(wide_path)
print(f"\nSaved wide matrix to: {wide_path}")
print(f"File size: {os.path.getsize(wide_path)} bytes")
print(f"Matrix shape: {df_wide.shape} (genes x cancer types)")

# QC Summary
print(f"\n{'='*80}")
print("QC SUMMARY")
print(f"{'='*80}")

# N stats per study
n_per_study = df_long.groupby('studyId')['N'].first()
print(f"\nDenominator N per study:")
print(f"  Min:    {n_per_study.min()}")
print(f"  Median: {n_per_study.median():.0f}")
print(f"  Max:    {n_per_study.max()}")
print(f"  Total:  {n_per_study.sum()}")

# Top altered genes across all studies
print(f"\nMean any_alter_freq across all studies, by gene:")
mean_freq = df_long.groupby('geneSymbol')['any_alter_freq'].mean().sort_values(ascending=False)
for g, v in mean_freq.items():
    print(f"  {g:<12} {v:.4f}")

# Example rows
print(f"\nExample rows (first 10):")
print(df_long.head(10)[['studyId', 'cancerTypeId', 'geneSymbol', 'N', 'mut_n', 'any_cna_n', 'any_alter_n', 'any_alter_freq']].to_string(index=False))

# Wide matrix preview
print(f"\nWide matrix preview (first 5 genes, first 5 cancer types):")
print(df_wide.iloc[:5, :5].round(4).to_string())


Profile map: 32 studies
Intersection map: 32 studies
[1/32] acc_tcga_pan_can_atlas_2018 (N=89) ... 

MUT=26 

ERR(HTTPError), retry in 2s... 

ERR(HTTPError), retry in 4s... 

CNA=1335 

...

ERR(HTTPError), retry in 2s... 

ERR(HTTPError), retry in 4s... 

CNA=1200 

OK

PROCESSING COMPLETE
Studies processed: 32
Studies skipped (N=0): 0
Total rows in long table: 480

Saved long-format table to: /home/user/tcga_pancan_miRNA_biogenesis_alteration_freqs_long.csv
File size: 114513 bytes



Saved wide matrix to: /home/user/tcga_pancan_miRNA_biogenesis_any_alter_freq_matrix.csv
File size: 8493 bytes
Matrix shape: (15, 30) (genes x cancer types)

QC SUMMARY

Denominator N per study:
  Min:    36
  Median: 324
  Max:    1052
  Total:  10195

Mean any_alter_freq across all studies, by gene:
  TP53         0.5507
  AGO2         0.4816
  SMAD4        0.4117
  SMAD2        0.4084
  DGCR8        0.4051
  LIN28B       0.3880
  DICER1       0.3767
  LIN28A       0.3755
  DROSHA       0.3713
  AGO4         0.3492
  AGO3         0.3490
  AGO1         0.3487
  XPO5         0.3422
  TARBP2       0.3077
  PRKRA        0.2617

Example rows (first 10):
                    studyId cancerTypeId geneSymbol  N  mut_n  any_cna_n  any_alter_n  any_alter_freq
acc_tcga_pan_can_atlas_2018          acc     DROSHA 89      2         66           66        0.741573
acc_tcga_pan_can_atlas_2018          acc      DGCR8 89      1         51           52        0.584270
acc_tcga_pan_can_atlas_2018        

## Step 7: Create final gene-by-cancer alteration-frequency matrices (with class breakdown) plus ranked highlights and TCGA coverage notes

Use the already-computed long-format TCGA PanCancer Atlas alteration-frequency table (32 studies × 15 genes) to (i) pivot gene×tumour-type matrices for overall and per-class frequencies, (ii) rank genes and tumour types by alteration burden, and (iii) flag tumour types with small mutation∩CNA denominators where TCGA coverage is limited and COSMIC supplementation is most justified.

In [10]:

import pandas as pd
import numpy as np

# Load data
df_long = pd.read_csv("/home/user/tcga_pancan_miRNA_biogenesis_alteration_freqs_long.csv")
df_sample_summary = pd.read_csv("/home/user/tcga_pancan_atlas_sample_set_summary.csv")

print("=" * 80)
print("DATA VERIFICATION")
print("=" * 80)

print(f"\nLong table shape: {df_long.shape}")
print(f"Long table columns: {list(df_long.columns)}")
n_studies = df_long['studyId'].nunique()
n_genes = df_long['geneSymbol'].nunique()
print(f"Unique studies: {n_studies}, Unique genes: {n_genes}")
print(f"Expected rows (32×15): {32*15}, Actual: {len(df_long)}")

# Check required columns
required_cols = ['studyId', 'geneSymbol', 'N', 'mut_n', 'mut_freq', 'amp_n', 'amp_freq',
                 'gain_n', 'gain_freq', 'shallow_del_n', 'shallow_del_freq', 
                 'deep_del_n', 'deep_del_freq', 'any_cna_freq', 'any_alter_freq']
missing = [c for c in required_cols if c not in df_long.columns]
print(f"Missing required columns: {missing if missing else 'None'}")

print(f"\nSample summary shape: {df_sample_summary.shape}")
print(f"Sample summary columns: {list(df_sample_summary.columns)}")

# Derive TCGA acronym from studyId
df_long['tumour_acronym'] = df_long['studyId'].str.replace('_tcga_pan_can_atlas_2018', '', regex=False).str.upper()
print(f"\nTumour acronyms ({df_long['tumour_acronym'].nunique()}): {sorted(df_long['tumour_acronym'].unique())}")

# Also add to sample summary
df_sample_summary['tumour_acronym'] = df_sample_summary['studyId'].str.replace('_tcga_pan_can_atlas_2018', '', regex=False).str.upper()

# ============================================================
# 1) Pivot and save gene×tumour matrices for each frequency metric
# ============================================================
print("\n" + "=" * 80)
print("1) PIVOTING & SAVING GENE × TUMOUR MATRICES")
print("=" * 80)

TARGET_GENES_ORDER = ['DROSHA','DGCR8','DICER1','TARBP2','PRKRA','XPO5',
                      'AGO1','AGO2','AGO3','AGO4','LIN28A','LIN28B',
                      'SMAD2','SMAD4','TP53']

freq_metrics = {
    'any_alter_freq': 'any_alter_freq',
    'mut_freq': 'mut_freq',
    'any_cna_freq': 'any_cna_freq',
    'amp_freq': 'amp_freq',
    'gain_freq': 'gain_freq',
    'shallow_del_freq': 'shallow_del_freq',
    'deep_del_freq': 'deep_del_freq',
}

for metric_name, col_name in freq_metrics.items():
    df_pivot = df_long.pivot_table(
        index='geneSymbol',
        columns='tumour_acronym',
        values=col_name,
        aggfunc='first'
    )
    # Reorder rows
    df_pivot = df_pivot.reindex([g for g in TARGET_GENES_ORDER if g in df_pivot.index])
    # Sort columns alphabetically
    df_pivot = df_pivot[sorted(df_pivot.columns)]
    
    out_path = f"/home/user/tcga_pancan_miRNA_biogenesis_{metric_name}_matrix.csv"
    df_pivot.to_csv(out_path)
    print(f"  Saved {metric_name} matrix ({df_pivot.shape}) -> {out_path}")

# ============================================================
# 2) Final long-format table with tumour acronym
# ============================================================
print("\n" + "=" * 80)
print("2) FINAL LONG-FORMAT TABLE")
print("=" * 80)

# Build final long table with selected columns
final_cols = ['tumour_acronym', 'studyId', 'geneSymbol', 'entrezGeneId', 'N',
              'mut_n', 'mut_freq', 'amp_n', 'amp_freq', 'gain_n', 'gain_freq',
              'shallow_del_n', 'shallow_del_freq', 'deep_del_n', 'deep_del_freq',
              'any_cna_n', 'any_cna_freq', 'any_alter_n', 'any_alter_freq',
              'mut_only_n', 'cna_only_n', 'mut_and_cna_n']

# Add studyName if present
if 'studyName' in df_long.columns:
    final_cols.insert(2, 'studyName')
elif 'cancerTypeId' in df_long.columns:
    # Use cancerTypeId as fallback
    pass

available_cols = [c for c in final_cols if c in df_long.columns]
df_final_long = df_long[available_cols].sort_values(['tumour_acronym', 'geneSymbol']).reset_index(drop=True)

final_long_path = "/home/user/tcga_pancan_miRNA_biogenesis_alteration_freqs_long_FINAL.csv"
df_final_long.to_csv(final_long_path, index=False)
print(f"Saved final long table ({df_final_long.shape}) -> {final_long_path}")
print(f"Columns: {list(df_final_long.columns)}")
print(f"\nHead (5 rows):")
print(df_final_long.head().to_string(index=False))

# ============================================================
# 3a) Gene rank summary by mean any_alter_freq
# ============================================================
print("\n" + "=" * 80)
print("3a) GENE RANK BY MEAN any_alter_freq")
print("=" * 80)

gene_rank = df_long.groupby('geneSymbol').agg(
    mean_any_alter_freq=('any_alter_freq', 'mean'),
    mean_mut_freq=('mut_freq', 'mean'),
    mean_any_cna_freq=('any_cna_freq', 'mean'),
    mean_amp_freq=('amp_freq', 'mean'),
    mean_deep_del_freq=('deep_del_freq', 'mean'),
    max_any_alter_freq=('any_alter_freq', 'max'),
    n_studies=('studyId', 'nunique'),
).sort_values('mean_any_alter_freq', ascending=False).reset_index()
gene_rank.index = range(1, len(gene_rank) + 1)
gene_rank.index.name = 'rank'

gene_rank_path = "/home/user/miRNA_biogenesis_gene_rank_summary.csv"
gene_rank.to_csv(gene_rank_path)
print(gene_rank.to_string())

# ============================================================
# 3b) Top 5 tumour types per gene by any_alter_freq
# ============================================================
print("\n" + "=" * 80)
print("3b) TOP 5 TUMOUR TYPES PER GENE")
print("=" * 80)

top5_rows = []
for g in TARGET_GENES_ORDER:
    gdf = df_long[df_long['geneSymbol'] == g].sort_values('any_alter_freq', ascending=False).head(5)
    for rank_i, (_, r) in enumerate(gdf.iterrows(), 1):
        top5_rows.append({
            'geneSymbol': g,
            'rank': rank_i,
            'tumour_acronym': r['tumour_acronym'],
            'N': int(r['N']),
            'any_alter_freq': round(r['any_alter_freq'], 4),
            'mut_freq': round(r['mut_freq'], 4),
            'any_cna_freq': round(r['any_cna_freq'], 4),
            'amp_freq': round(r['amp_freq'], 4),
            'deep_del_freq': round(r['deep_del_freq'], 4),
        })

df_top5 = pd.DataFrame(top5_rows)
top5_path = "/home/user/miRNA_biogenesis_top5_tumours_per_gene.csv"
df_top5.to_csv(top5_path, index=False)
print(f"Saved top 5 tumours per gene ({len(df_top5)} rows) -> {top5_path}")

# Print preview for each gene (top 3)
for g in TARGET_GENES_ORDER:
    subset = df_top5[df_top5['geneSymbol'] == g].head(3)
    tops = ", ".join([f"{r['tumour_acronym']}({r['any_alter_freq']:.3f})" for _, r in subset.iterrows()])
    print(f"  {g:<12}: {tops}")

# ============================================================
# 3c) Tumour ranking by mean any_alter_freq across 14 biogenesis genes (excl TP53)
# ============================================================
print("\n" + "=" * 80)
print("3c) TUMOUR RANKING (EXCLUDING TP53)")
print("=" * 80)

biogenesis_genes_no_tp53 = [g for g in TARGET_GENES_ORDER if g != 'TP53']
df_no_tp53 = df_long[df_long['geneSymbol'].isin(biogenesis_genes_no_tp53)]

tumour_rank = df_no_tp53.groupby('tumour_acronym').agg(
    mean_any_alter_freq_excl_TP53=('any_alter_freq', 'mean'),
    mean_mut_freq_excl_TP53=('mut_freq', 'mean'),
    mean_any_cna_freq_excl_TP53=('any_cna_freq', 'mean'),
    N=('N', 'first'),
    n_genes=('geneSymbol', 'nunique'),
).sort_values('mean_any_alter_freq_excl_TP53', ascending=False).reset_index()
tumour_rank.index = range(1, len(tumour_rank) + 1)
tumour_rank.index.name = 'rank'

tumour_rank_path = "/home/user/miRNA_biogenesis_top_tumours_excluding_TP53.csv"
tumour_rank.to_csv(tumour_rank_path)
print(f"Saved tumour ranking ({len(tumour_rank)} rows) -> {tumour_rank_path}")
print("\nTop 10 tumour types by mean any_alter_freq (excl TP53):")
print(tumour_rank.head(10).to_string())

# ============================================================
# 4) Coverage note: low N tumours
# ============================================================
print("\n" + "=" * 80)
print("4) LOW COVERAGE TUMOUR TYPES")
print("=" * 80)

# Get N per tumour from df_long (one value per study)
n_per_tumour = df_long.groupby(['tumour_acronym', 'studyId'])['N'].first().reset_index()
n_per_tumour = n_per_tumour.sort_values('N')

# Tumours with N < 100
low_n = n_per_tumour[n_per_tumour['N'] < 100]
# Bottom 5 by N
bottom5 = n_per_tumour.head(5)

coverage_lines = []
coverage_lines.append("=" * 70)
coverage_lines.append("TCGA PanCancer Atlas: Low-Coverage Tumour Types for miRNA Biogenesis Analysis")
coverage_lines.append("=" * 70)
coverage_lines.append(f"\nDenominator: intersection of sequenced and CNA sample lists")
coverage_lines.append(f"Total tumour types: {n_per_tumour['tumour_acronym'].nunique()}")
coverage_lines.append(f"Total samples across all tumour types: {int(n_per_tumour['N'].sum())}")
coverage_lines.append("")
coverage_lines.append("-" * 70)
coverage_lines.append("Tumour types with N < 100:")
coverage_lines.append("-" * 70)
if len(low_n) > 0:
    for _, r in low_n.iterrows():
        coverage_lines.append(f"  {r['tumour_acronym']:<15} N = {int(r['N']):>4}  ({r['studyId']})")
else:
    coverage_lines.append("  None — all tumour types have N >= 100")

coverage_lines.append("")
coverage_lines.append("-" * 70)
coverage_lines.append("Bottom 5 tumour types by N:")
coverage_lines.append("-" * 70)
for _, r in bottom5.iterrows():
    coverage_lines.append(f"  {r['tumour_acronym']:<15} N = {int(r['N']):>4}  ({r['studyId']})")

coverage_lines.append("")
coverage_lines.append("-" * 70)
coverage_lines.append("Note: Low sample counts may reduce statistical power for detecting")
coverage_lines.append("low-frequency alterations. Interpret frequencies from these tumour types")
coverage_lines.append("with caution.")
coverage_lines.append("-" * 70)

coverage_text = "\n".join(coverage_lines)

coverage_path = "/home/user/tcga_low_coverage_tumours_for_miRNA_biogenesis.txt"
with open(coverage_path, 'w') as f:
    f.write(coverage_text)
print(f"Saved coverage notes -> {coverage_path}")
print()
print(coverage_text)

# ============================================================
# Final file listing
# ============================================================
print("\n" + "=" * 80)
print("ALL OUTPUT FILES")
print("=" * 80)
import os
output_files = [
    final_long_path,
    gene_rank_path,
    top5_path,
    tumour_rank_path,
    coverage_path,
]
# Add matrix files
for metric_name in freq_metrics:
    output_files.append(f"/home/user/tcga_pancan_miRNA_biogenesis_{metric_name}_matrix.csv")

for fp in sorted(output_files):
    sz = os.path.getsize(fp)
    if sz > 1024:
        sz_str = f"{sz/1024:.1f} KB"
    else:
        sz_str = f"{sz} B"
    print(f"  {os.path.basename(fp):<65} {sz_str:>10}")


DATA VERIFICATION

Long table shape: (480, 23)
Long table columns: ['studyId', 'cancerTypeId', 'studyName', 'geneSymbol', 'entrezGeneId', 'N', 'mut_n', 'amp_n', 'gain_n', 'shallow_del_n', 'deep_del_n', 'any_cna_n', 'any_alter_n', 'mut_only_n', 'cna_only_n', 'mut_and_cna_n', 'mut_freq', 'amp_freq', 'gain_freq', 'shallow_del_freq', 'deep_del_freq', 'any_cna_freq', 'any_alter_freq']
Unique studies: 32, Unique genes: 15
Expected rows (32×15): 480, Actual: 480
Missing required columns: None

Sample summary shape: (32, 14)
Sample summary columns: ['studyId', 'cancerTypeId', 'studyName', 'sampleCount_all', 'sampleCount_sequenced', 'sampleCount_cna', 'intersection_sequenced_cna_n', 'cnaseq_sampleListId', 'cnaseq_n', 'threeway_sampleListId', 'threeway_n', 'ratio_intersection_over_min_seq_cna', 'ratio_cnaseq_over_intersection', 'ratio_threeway_over_intersection']

Tumour acronyms (32): ['ACC', 'BLCA', 'BRCA', 'CESC', 'CHOL', 'COADREAD', 'DLBC', 'ESCA', 'GBM', 'HNSC', 'KICH', 'KIRC', 'KIRP', 'LAM